# Stage 03 — baselines: persistence, HAR, GARCH(1,1)

Three baselines under 11-fold walk-forward CV (expanding train, 252-day test
blocks, 21-day embargo so no h=21 label overlaps a test window): persistence
(trailing rv_h copied forward), HAR (OLS on rv1/rv5/rv21, Corsi 2009), and
GARCH(1,1) — the likelihood is hand-rolled and validated against the `arch`
package (parameters agree to 3–4 decimals).

Fold-mean h=21 RMSE: persistence 0.0056, HAR 0.0049, GARCH 0.0049. The fact
reused by every later stage: f07 (Feb 2022–Feb 2023) is the one fold where
every fitted model loses to persistence (HAR −25.1%), the signature of a
P(Y|X) break rather than a volatility spike.


In [ ]:
# Setup: enter the project root and import shared utilities.
import os, sys
import numpy as np
import pandas as pd

# Environment bootstrap: Colab (mount Drive) or a local checkout.
try:
    from google.colab import drive
    drive.mount("/content/drive")
    os.chdir("/content/drive/MyDrive/volatility-forecast")
except ModuleNotFoundError:
    p = os.path.abspath(os.getcwd())          # find the repo root locally
    while not os.path.isdir(os.path.join(p, "src")):
        parent = os.path.dirname(p)
        if parent == p:
            raise FileNotFoundError("repo root with src/ not found")
        p = parent
    os.chdir(p)

for m in list(sys.modules):   # notebooks cache imports; drop stale src modules
    if m.startswith("src"):
        del sys.modules[m]

from src.splits import walk_forward_splits
from src.metrics import rmse
from src.har import fit_har, predict_har


In [2]:
# Load the stage-02 dataset and build the shared work frame:
# drop rows where any feature or target is NaN (21-row rolling warmup at
# the head, 21-row forward-target gap at the tail).
df = pd.read_csv("data/processed/dataset.csv", index_col=0, parse_dates=True)

feat = ['qqq_ret','hyg_ret','lqd_ret','tlt_ret','gld_ret','vix_lvl','vix_chg',
        'tnx_lvl','tnx_chg','irx_lvl','irx_chg','slope_lvl','slope_chg',
        'credit_lvl','credit_chg','rv1','rv5','rv21']
tgt = ['y_rv1','y_rv5','y_rv21']

work = df.dropna(subset=feat + tgt)   # shared frame across all horizons (y_rv21 binds)
print("full:", len(df), "working:", len(work),
      "| head trim:", df.index.get_loc(work.index[0]),
      "| tail trim:", len(df) - 1 - df.index.get_loc(work.index[-1]))

folds = list(walk_forward_splits(len(work)))
print("folds:", len(folds))
for k, (tr, te) in enumerate(folds):
    gap = te[0] - tr[-1] - 1
    print(f"f{k:02d} train[0:{len(tr):4d}] {work.index[0].date()}->{work.index[tr[-1]].date()}"
          f" | gap {gap} | test {work.index[te[0]].date()}->{work.index[te[-1]].date()} n={len(te)}")


full: 3873 working: 3831 | head trim: 21 | tail trim: 21
folds: 11
f00 train[0: 987] 2011-02-02->2015-01-05 | gap 21 | test 2015-02-05->2016-02-04 n=252
f01 train[0:1239] 2011-02-02->2016-01-05 | gap 21 | test 2016-02-05->2017-02-03 n=252
f02 train[0:1491] 2011-02-02->2017-01-04 | gap 21 | test 2017-02-06->2018-02-05 n=252
f03 train[0:1743] 2011-02-02->2018-01-04 | gap 21 | test 2018-02-06->2019-02-06 n=252
f04 train[0:1995] 2011-02-02->2019-01-07 | gap 21 | test 2019-02-07->2020-02-06 n=252
f05 train[0:2247] 2011-02-02->2020-01-07 | gap 21 | test 2020-02-07->2021-02-05 n=252
f06 train[0:2499] 2011-02-02->2021-01-06 | gap 21 | test 2021-02-08->2022-02-04 n=252
f07 train[0:2751] 2011-02-02->2022-01-05 | gap 21 | test 2022-02-07->2023-02-07 n=252
f08 train[0:3003] 2011-02-02->2023-01-06 | gap 21 | test 2023-02-08->2024-02-08 n=252
f09 train[0:3255] 2011-02-02->2024-01-09 | gap 21 | test 2024-02-09->2025-02-11 n=252
f10 train[0:3507] 2011-02-02->2025-01-10 | gap 21 | test 2025-02-12->2026

In [3]:
# persistence: forward-h vol predicted by trailing-h vol (same column family)
pairs = {1:('rv1','y_rv1'), 5:('rv5','y_rv5'), 21:('rv21','y_rv21')}

rows = []
for h,(xcol,ycol) in pairs.items():
    per_fold = []
    for k,(tr,te) in enumerate(folds):
        yhat = work[xcol].values[te]   # prediction = trailing rv_h at t
        ytru = work[ycol].values[te]   # target = forward rv_h
        fold_rmse = np.sqrt(np.mean((yhat-ytru)**2))   # renamed
        per_fold.append(fold_rmse)
    per_fold = np.array(per_fold)
    rows.append({'h':h,'rmse_mean':per_fold.mean(),'rmse_std':per_fold.std(),
                 'rmse_min':per_fold.min(),'rmse_max':per_fold.max()})
    print(f"h={h:2d} | per-fold RMSE: " + " ".join(f"{v:.4f}" for v in per_fold))

base = pd.DataFrame(rows).set_index('h')
print()
print(base.round(4))

h= 1 | per-fold RMSE: 0.0109 0.0073 0.0069 0.0129 0.0083 0.0185 0.0110 0.0174 0.0096 0.0108 0.0137
h= 5 | per-fold RMSE: 0.0062 0.0048 0.0046 0.0068 0.0044 0.0114 0.0048 0.0080 0.0044 0.0046 0.0092
h=21 | per-fold RMSE: 0.0050 0.0041 0.0036 0.0061 0.0050 0.0146 0.0043 0.0045 0.0022 0.0038 0.0084

    rmse_mean  rmse_std  rmse_min  rmse_max
h                                          
1      0.0116    0.0036    0.0069    0.0185
5      0.0063    0.0022    0.0044    0.0114
21     0.0056    0.0032    0.0022    0.0146


In [4]:
# Fold-structure sanity checks: expanding train from row 0, a 21-row
# embargo gap, and no h=21 label overlapping the test window.
H_MAX = 21
for k, (tr, te) in enumerate(folds):
    last_train, first_test = tr[-1], te[0]
    assert tr[0] == 0,                        f"f{k}: train does not start at 0"
    assert first_test - last_train - 1 == 21, f"f{k}: gap != embargo"
    assert last_train + H_MAX < first_test,   f"f{k}: label leaks into test"
print("all folds: expanding from 0, gap=21, no label leak")


all folds: expanding from 0, gap=21, no label leak


In [5]:
FEATURES = ["rv1", "rv5", "rv21"]
HORIZONS = {"h1": "y_rv1", "h5": "y_rv5", "h21": "y_rv21"}

folds = list(walk_forward_splits(len(work)))
har_results = {h: [] for h in HORIZONS}   # per-horizon list of per-fold RMSE

for h_name, y_col in HORIZONS.items():
    for tr, te in folds:
        X_tr = work[FEATURES].iloc[tr].values
        y_tr = work[y_col].iloc[tr].values
        X_te = work[FEATURES].iloc[te].values
        y_te = work[y_col].iloc[te].values

        beta = fit_har(X_tr, y_tr)
        yhat = predict_har(beta, X_te)
        har_results[h_name].append(rmse(y_te, yhat))

for h_name in HORIZONS:
    arr = np.array(har_results[h_name])
    print(f"HAR {h_name:3s}  mean {arr.mean():.4f}  std {arr.std():.4f}  "
          f"min {arr.min():.4f}  max {arr.max():.4f}")

HAR h1   mean 0.0087  std 0.0028  min 0.0055  max 0.0148
HAR h5   mean 0.0055  std 0.0022  min 0.0035  max 0.0111
HAR h21  mean 0.0049  std 0.0026  min 0.0020  max 0.0122


In [6]:
H_COL = "y_rv21"          # focus on h=21, where the crisis story is sharpest
FEATURES = ["rv1", "rv5", "rv21"]
PERSIST_COL = "rv21"      # persistence: copy trailing rv21 as the forecast

folds = list(walk_forward_splits(len(work)))

print(f"{'fold':>4} {'persist':>9} {'HAR':>9} {'HAR win':>9}")
for k, (tr, te) in enumerate(folds):
    y_te = work[H_COL].iloc[te].values

    # persistence: yhat = trailing rv21 (no fit)
    yhat_p = work[PERSIST_COL].iloc[te].values
    rmse_p = rmse(y_te, yhat_p)

    # HAR: fit on train, predict on test
    X_tr = work[FEATURES].iloc[tr].values
    y_tr = work[H_COL].iloc[tr].values
    X_te = work[FEATURES].iloc[te].values
    beta = fit_har(X_tr, y_tr)
    yhat_h = predict_har(beta, X_te)
    rmse_h = rmse(y_te, yhat_h)

    win = (rmse_p - rmse_h) / rmse_p * 100   # positive = HAR better
    flag = "" if win > 0 else "  <-- LOST"
    print(f"f{k:02d}  {rmse_p:9.4f} {rmse_h:9.4f} {win:8.1f}%{flag}")

fold   persist       HAR   HAR win
f00     0.0050    0.0043     14.6%
f01     0.0041    0.0033     19.1%
f02     0.0036    0.0034      7.4%
f03     0.0061    0.0052     14.3%
f04     0.0050    0.0040     19.8%
f05     0.0146    0.0122     16.7%
f06     0.0043    0.0040      6.8%
f07     0.0045    0.0056    -25.1%  <-- LOST
f08     0.0022    0.0020     11.9%
f09     0.0038    0.0031     18.4%
f10     0.0084    0.0068     18.9%


In [7]:
from scipy.optimize import minimize


def garch11_nll(params, r):
    """
    Negative log-likelihood of a Gaussian GARCH(1,1).

    Model:
        r_t      = mu + eps_t,  eps_t = sigma_t * z_t,  z_t ~ N(0, 1)
        sigma2_t = omega + alpha * eps_{t-1}^2 + beta * sigma2_{t-1}

    params = [mu, omega, alpha, beta]
    r      = 1D array of returns (percent scale recommended)
    """
    mu, omega, alpha, beta = params

    # Constraint guard: penalize invalid region so the optimizer backs away.
    if omega <= 0 or alpha < 0 or beta < 0 or (alpha + beta) >= 1.0:
        return 1e10

    eps = r - mu
    T = len(r)
    sigma2 = np.empty(T)
    sigma2[0] = np.var(eps)  # initialize recursion with sample variance

    for t in range(1, T):
        sigma2[t] = omega + alpha * eps[t - 1] ** 2 + beta * sigma2[t - 1]

    # Conditional Gaussian log-likelihood (prediction-error decomposition).
    ll = -0.5 * np.sum(np.log(2.0 * np.pi) + np.log(sigma2) + eps ** 2 / sigma2)
    return -ll

ret_raw = df["qqq_ret"].dropna()        # dropna on the return column ONLY

# Scale sniff: decimal (~0.01) vs percent (~1.0)?
print(f"qqq_ret  n={len(ret_raw)}  "
      f"std={ret_raw.std():.5f}  "
      f"abs_max={ret_raw.abs().max():.5f}")

# Daily index return std should be ~0.01 (decimal) or ~1.0 (percent).
# We want PERCENT scale for GARCH optimizer stability.
if ret_raw.std() < 0.1:          # looks like decimal
    ret = ret_raw * 100.0
    print("-> detected DECIMAL scale, multiplied by 100 to get percent")
else:                            # already percent
    ret = ret_raw.copy()
    print("-> detected PERCENT scale, used as-is")

r = ret.values

# --- Fit GARCH(1,1) by MLE ---------------------------------------------
x0 = [r.mean(), 0.05, 0.05, 0.90]   # [mu, omega, alpha, beta], feasible start
res = minimize(
    garch11_nll, x0, args=(r,),
    method="Nelder-Mead",
    options={"xatol": 1e-8, "fatol": 1e-8, "maxiter": 20000},
)

mu, omega, alpha, beta = res.x
persistence = alpha + beta
uncond_var = omega / (1.0 - persistence)
uncond_vol_daily = np.sqrt(uncond_var)              # percent (r is percent)
uncond_vol_annual = uncond_vol_daily * np.sqrt(252)

print(f"converged       : {res.success}")
print(f"n returns       : {len(r)}")
print(f"log-likelihood  : {-res.fun:,.2f}")
print(f"mu              : {mu:.5f}")
print(f"omega           : {omega:.6f}")
print(f"alpha (news)    : {alpha:.4f}")
print(f"beta  (memory)  : {beta:.4f}")
print(f"persistence a+b : {persistence:.4f}   (<1 for stationarity)")
print(f"uncond vol/day  : {uncond_vol_daily:.3f}%")
print(f"uncond vol/yr   : {uncond_vol_annual:.1f}%")

qqq_ret  n=3872  std=0.01304  abs_max=0.12759
-> detected DECIMAL scale, multiplied by 100 to get percent
converged       : True
n returns       : 3872
log-likelihood  : -5,891.18
mu              : 0.10189
omega           : 0.049025
alpha (news)    : 0.1325
beta  (memory)  : 0.8386
persistence a+b : 0.9711   (<1 for stationarity)
uncond vol/day  : 1.303%
uncond vol/yr   : 20.7%


In [8]:
!pip install arch -q

from arch import arch_model

# Same series we fit by hand: percent-scale QQQ returns (ret is a pd.Series).
am = arch_model(ret, mean="Constant", vol="GARCH", p=1, q=1, dist="normal")
fit = am.fit(disp="off")

# arch parameter names: mu, omega, alpha[1], beta[1]
p = fit.params
mu_a    = p["mu"]
omega_a = p["omega"]
alpha_a = p["alpha[1]"]
beta_a  = p["beta[1]"]
pers_a  = alpha_a + beta_a

print("            hand-rolled     arch")
print(f"mu        : {mu:>10.5f}   {mu_a:>10.5f}")
print(f"omega     : {omega:>10.6f}   {omega_a:>10.6f}")
print(f"alpha     : {alpha:>10.4f}   {alpha_a:>10.4f}")
print(f"beta      : {beta:>10.4f}   {beta_a:>10.4f}")
print(f"persist.  : {persistence:>10.4f}   {pers_a:>10.4f}")
print(f"loglik    : {-res.fun:>10.2f}   {fit.loglikelihood:>10.2f}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.3/981.3 kB 42.3 MB/s eta 0:00:00
            hand-rolled     arch
mu        :    0.10189      0.10204
omega     :   0.049025     0.049041
alpha     :     0.1325       0.1323
beta      :     0.8386       0.8388
persist.  :     0.9711       0.9711
loglik    :   -5891.18     -5889.28


In [9]:
# ==========================================================================
# GARCH(1,1) walk-forward (manual recursion, leak-free, full control)
# Train per fold -> fix params -> roll sigma2 with observed returns ->
# h-step mean-revert forecast -> aggregate to match y_rv{h} targets
# ==========================================================================
from arch import arch_model

# Percent-scale returns aligned to the work-frame index.
ret_full = (df["qqq_ret"] * 100.0).dropna()
ret_vals = ret_full.values
ret_pos = {ts: i for i, ts in enumerate(ret_full.index)}
work_to_ret = np.array([ret_pos[ts] for ts in work.index])   # work row -> ret row


def garch_sigma2_path(params, r):
    """Roll conditional variance over r with fixed params (percent^2).
    sigma2[t] uses info up to t-1 (returns r[:t]); leak-free 1-step recursion."""
    mu, omega, alpha, beta = params
    eps = r - mu
    T = len(r)
    s2 = np.empty(T)
    s2[0] = np.var(eps)
    for t in range(1, T):
        s2[t] = omega + alpha * eps[t - 1] ** 2 + beta * s2[t - 1]
    return s2, eps


HORIZONS = {"h1": ("y_rv1", 1), "h5": ("y_rv5", 5), "h21": ("y_rv21", 21)}
garch_results = {h: [] for h in HORIZONS}
folds = list(walk_forward_splits(len(work)))

for k, (tr, te) in enumerate(folds):
    # --- fit on train returns to get parameters (use arch, validated) -----
    train_end = work_to_ret[tr[-1]]
    r_train = ret_vals[: train_end + 1]
    am = arch_model(r_train, mean="Constant", vol="GARCH",
                    p=1, q=1, dist="normal", rescale=False)
    fit = am.fit(disp="off")
    pp = fit.params
    params = (pp["mu"], pp["omega"], pp["alpha[1]"], pp["beta[1]"])
    mu, omega, alpha, beta = params
    uncond = omega / (1.0 - alpha - beta)        # long-run variance (percent^2)
    decay = alpha + beta                         # persistence

    # --- roll sigma2 over ALL returns with fixed params (observed-based) --
    s2, eps = garch_sigma2_path(params, ret_vals)

    # --- for each test row t: build h-step variance forecast --------------
    yhat = {h_name: [] for h_name in HORIZONS}
    for t_ret in work_to_ret[te]:
        # 1-step ahead from t (uses eps[t], s2[t]; both info up to t)
        s2_1 = omega + alpha * eps[t_ret] ** 2 + beta * s2[t_ret]
        # k-step mean-revert forecasts, k = 1..H_MAX
        ks = np.arange(1, 22)
        s2_k = uncond + decay ** (ks - 1) * (s2_1 - uncond)   # percent^2
        for h_name, (y_col, h) in HORIZONS.items():
            var_h = s2_k[:h].mean()              # avg conditional var over t+1..t+h
            yhat[h_name].append(np.sqrt(var_h) / 100.0)       # -> decimal RV

    for h_name, (y_col, h) in HORIZONS.items():
        y_te = work[y_col].values[te]
        garch_results[h_name].append(rmse(y_te, np.array(yhat[h_name])))

for h_name in HORIZONS:
    arr = np.array(garch_results[h_name])
    print(f"GARCH {h_name:3s}  mean {arr.mean():.4f}  std {arr.std():.4f}  "
          f"min {arr.min():.4f}  max {arr.max():.4f}")

GARCH h1   mean 0.0090  std 0.0026  min 0.0061  max 0.0145
GARCH h5   mean 0.0055  std 0.0020  min 0.0036  max 0.0104
GARCH h21  mean 0.0049  std 0.0025  min 0.0021  max 0.0117


In [10]:
# ==========================================================================
# Per-fold comparison at h=21: persistence vs HAR vs GARCH
# (GARCH per-fold RMSE already computed in garch_results["h21"])
# ==========================================================================
H_COL = "y_rv21"
FEATURES = ["rv1", "rv5", "rv21"]
PERSIST_COL = "rv21"

folds = list(walk_forward_splits(len(work)))
g_h21 = garch_results["h21"]            # per-fold GARCH RMSE, same fold order

print(f"{'fold':>4} {'persist':>9} {'HAR':>9} {'GARCH':>9}   "
      f"{'HAR win':>8} {'GAR win':>8}  best")
for k, (tr, te) in enumerate(folds):
    y_te = work[H_COL].values[te]

    rmse_p = rmse(y_te, work[PERSIST_COL].values[te])

    X_tr, y_tr = work[FEATURES].iloc[tr].values, work[H_COL].iloc[tr].values
    beta = fit_har(X_tr, y_tr)
    rmse_h = rmse(y_te, predict_har(beta, work[FEATURES].iloc[te].values))

    rmse_g = g_h21[k]

    har_win = (rmse_p - rmse_h) / rmse_p * 100      # vs persistence
    gar_win = (rmse_p - rmse_g) / rmse_p * 100      # vs persistence
    best = min([("persist", rmse_p), ("HAR", rmse_h), ("GARCH", rmse_g)],
               key=lambda x: x[1])[0]

    print(f"f{k:02d}  {rmse_p:9.4f} {rmse_h:9.4f} {rmse_g:9.4f}   "
          f"{har_win:7.1f}% {gar_win:7.1f}%  {best}")

fold   persist       HAR     GARCH    HAR win  GAR win  best
f00     0.0050    0.0043    0.0042      14.6%    16.1%  GARCH
f01     0.0041    0.0033    0.0035      19.1%    15.1%  HAR
f02     0.0036    0.0034    0.0037       7.4%    -1.3%  HAR
f03     0.0061    0.0052    0.0051      14.3%    16.0%  GARCH
f04     0.0050    0.0040    0.0039      19.8%    22.4%  GARCH
f05     0.0146    0.0122    0.0117      16.7%    19.7%  GARCH
f06     0.0043    0.0040    0.0039       6.8%     9.4%  GARCH
f07     0.0045    0.0056    0.0051     -25.1%   -14.6%  persist
f08     0.0022    0.0020    0.0021      11.9%     4.5%  HAR
f09     0.0038    0.0031    0.0032      18.4%    15.8%  HAR
f10     0.0084    0.0068    0.0072      18.9%    14.0%  HAR
